# 03_unsup_ate.ipynb
## Purpose
UNSUP candidate discovery + semantic-context ranking; outputs per-segment candidate lists.

## Expected inputs
- `data/segments_index.csv`
- `preprocessed UNSUP text`

## Expected outputs
- `outputs/unsup_output.csv`

## Notes
- Keys are aligned using `seg_key = comment_id__seg_id`.

## 0. I/O configuration
Mato sure file `dataset_unsup.csv` is in the same folder with notebook or adjust the path below.
Required columns: `comment_id`, `seg_id`, `source`, `date` (optional), `seg_text_raw` (optional), **`unsup_filtered`**.

# 1) Setup

In [ ]:
# ============================================================
# 1) Setup (stable, torch-only)
# ============================================================
# Notes:
# - This pipeline focuses on UNSUP term extraction + LDA + clustering + ranking.


import os
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")  # aman walau TF tidak dipakai

import json, time, platform, re
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# gensim (LDA)
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# embeddings
import torch
from sentence_transformers import SentenceTransformer

# clustering + similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm
tqdm.pandas()

# POS filter
import stanza

print("Python:", platform.python_version())
print("Torch :", torch.__version__)
print("Gensim:", gensim.__version__)

#  2) Parameters

In [ ]:

# ============================================================
# 2) Centralized parameters (audit-ready)
# ============================================================
@dataclass
class Config:
    # I/O
    input_csv: str = "data\\dataset_unsup.csv"
    out_dir: str = "data\\outputs_unsup"

    # column teks
    text_col: str = "unsup_filtered"

    # LDA
    lda_topic_min: int = 5
    lda_topic_max: int = 25
    lda_step: int = 5
    lda_passes: int = 20
    lda_random_state: int = 42

    lda_chunksize: int = 2000  # batch size dokumen untuk update LDA (perf)
    # merge vektor
    gamma_lda: float = 0.5  # bobot vecLDA sebelum concat dengan sentence embedding

    # sentence embedding (pakai satu model saja agar konsisten for cosine ranking)
    st_model_name: str = "distiluse-base-multilingual-cased-v2"
    st_batch_size: int = 64

    # autoencoder
    ae_latent_dim: int = 64
    ae_hidden_dim: int = 256
    ae_epochs: int = 30
    ae_batch_size: int = 128
    ae_patience: int = 4

    # clustering
    k_min: int = 2
    k_max: int = 15
    k_random_state: int = 42

    # candidate terms per cluster
    top_terms_per_cluster: int = 200

    # ranking (per row)
    topn_rank: int = 50
    topn_keep: int = 10   # final top-N sebelum POS

    # stanza
    stanza_lang_primary: str = "id"
    stanza_lang_fallback: str = "en"

cfg = Config()
print(cfg)

# 3) Load data + validation

In [ ]:

# ============================================================
# 3) Load data + validation
# ============================================================
INP = Path(cfg.input_csv)
assert INP.exists(), f"File tidak ditemukan: {INP.resolve()}"

df = pd.read_csv(INP)
required = [cfg.text_col]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Kolom wajib tidak ada: {missing}. Kolom tersedia: {list(df.columns)}"

df[cfg.text_col] = df[cfg.text_col].fillna("").astype(str)
df = df[df[cfg.text_col].str.strip() != ""].copy()

# kunci join (for merge downstream)
join_cols = [c for c in ["comment_id","seg_id","source","date"] if c in df.columns]
print("Rows:", len(df))
print("Join cols:", join_cols)
df.head(3)

# 4) Utilitas audit + helper + totonisasi unsup_filtered

In [ ]:
# ============================================================
# 4) Utilitas audit + helper + totonisasi unsup_filtered
# ============================================================
import datetime

OUT_DIR = Path(cfg.out_dir)
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

_EDGE_PUNCT = re.compile(r"(^\W+|\W+$)", flags=re.UNICODE)

def simple_tokenize(s: str):
    """Tokenisasi ringan untuk `unsup_filtered`.
    Asumsi: sudah dipisah spasi. Kita tetap:
    - lower
    - trim
    - buang token kosong
    - buang punct di tepi token (mis. 'ckckck.' -> 'ckckck')
    """
    if s is None:
        return []
    toks = []
    for t in str(s).split():
        t = _EDGE_PUNCT.sub("", t.lower().strip())
        if t:
            toks.append(t)
    return toks

def build_doc_tokens(df: pd.DataFrame, text_col: str):
    return [simple_tokenize(x) for x in df[text_col].tolist()]

# --------- audit stoleton ----------
audit = {
    "created_at": datetime.datetime.now().isoformat(),
    "config": asdict(cfg),
    "dataset": {
        "input_csv": str(INP),
        "n_rows": int(len(df)),
        "columns": list(df.columns),
    },
    "timing_sec": {},
    "lda": {},
    "clustering": {},
    "ranking": {},
}

# --------- totonisasi sumber utama: unsup_filtered (cfg.text_col) ----------
t0 = time.time()
df["unsup_tokens"] = df[cfg.text_col].apply(simple_tokenize)

# audit toton stats
lens = df["unsup_tokens"].apply(len)
audit["tokenization"] = {
    "source_col": cfg.text_col,
    "avg_len": float(lens.mean()),
    "median_len": float(lens.median()),
    "min_len": int(lens.min()),
    "max_len": int(lens.max()),
    "rows_len0": int((lens == 0).sum()),
}
audit["timing_sec"]["tokenization"] = round(time.time() - t0, 3)

df[["unsup_filtered","unsup_tokens"]].head(3)

# 5) LDA: train + coherence sweep

In [ ]:
# ============================================================
# 5) LDA: train + coherence sweep (docs = totonisasi unsup_filtered)
# ============================================================
t0 = time.time()

docs = df["unsup_tokens"].tolist()

dictionary = corpora.Dictionary(docs)
# filter ekstrem (sesuaikan if perlu)
dictionary.filter_extremes(no_below=3, no_above=0.5)
corpus = [dictionary.doc2bow(d) for d in docs]

audit["lda"]["docs_source"] = "df['unsup_tokens'] (from unsup_filtered)"
audit["lda"]["dictionary_size"] = int(len(dictionary))
audit["lda"]["avg_bow_len"] = float(np.mean([len(b) for b in corpus])) if len(corpus) else 0.0

topic_list = list(range(cfg.lda_topic_min, cfg.lda_topic_max + 1, cfg.lda_step))
coh_scores = []

best_model = None
best_coh = -1.0
best_k = None

for k in tqdm(topic_list, desc="LDA topic sweep"):
    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=cfg.lda_random_state,
        chunksize=getattr(cfg, "lda_chunksize", 2000),
        passes=cfg.lda_passes,
        alpha="auto",
        eta="auto",
        per_word_topics=False,
    )
    cm = CoherenceModel(model=lda, texts=docs, dictionary=dictionary, coherence="c_v")
    coh = float(cm.get_coherence())
    coh_scores.append({"num_topics": int(k), "coherence_c_v": coh})
    if coh > best_coh:
        best_coh = coh
        best_k = k
        best_model = lda

print("Best LDA topics:", best_k, "coherence:", best_coh)

audit["lda"]["topic_sweep"] = coh_scores
audit["lda"]["best_num_topics"] = int(best_k) if best_k is not None else None
audit["lda"]["best_coherence_c_v"] = float(best_coh)

audit["timing_sec"]["lda"] = round(time.time() - t0, 3)

#  6) Vector LDA per document (dense)

In [ ]:

# ============================================================
# 6) Vector LDA per dokumen (dense)
# ============================================================
num_topics = audit["lda"]["best_num_topics"]
lda_model = best_model

def lda_dense_vec(lda_model, bow, num_topics: int):
    v = np.zeros(num_topics, dtype=np.float32)
    for t, p in lda_model.get_document_topics(bow, minimum_probability=0.0):
        v[int(t)] = float(p)
    return v

vec_lda = np.vstack([lda_dense_vec(lda_model, bow, num_topics) for bow in tqdm(corpus, desc="vecLDA")])
print("vec_lda:", vec_lda.shape)

df["vecLDA"] = list(vec_lda)  # optional: untuk debug

# 7) Sentence embeddings (SentenceTransformer)

In [ ]:

# ============================================================
# 7) Sentence embeddings (SentenceTransformer)
# ============================================================
st = SentenceTransformer(cfg.st_model_name)
sent_texts = df[cfg.text_col].tolist()

t0 = time.time()
sent_emb = st.encode(sent_texts, batch_size=cfg.st_batch_size, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
t1 = time.time()

sent_emb = sent_emb.astype(np.float32)
print("sentence_embeddings:", sent_emb.shape, "elapsed_sec:", round(t1-t0,2))

audit["embeddings"] = {
    "model": cfg.st_model_name,
    "shape": list(sent_emb.shape),
    "elapsed_sec": round(t1-t0,2),
}

# 8) Concat vecLDA + sentence embedding  → vec_ldabert

In [ ]:

# ============================================================
# 8) Concat vecLDA + sentence embedding  → vec_ldabert
# ============================================================
vec_lda_scaled = vec_lda * float(cfg.gamma_lda)
vec_ldabert = np.hstack([vec_lda_scaled, sent_emb]).astype(np.float32)

# scaling before AE
scaler = StandardScaler(with_mean=True, with_std=True)
vec_ldabert_sc = scaler.fit_transform(vec_ldabert).astype(np.float32)

print("vec_ldabert:", vec_ldabert.shape, "scaled:", vec_ldabert_sc.shape)

# 9) Autoencoder → latent vector

In [ ]:
 # autoencoder
import tensorflow as tf
try:
    import tf_keras  # legacy keras shim (optional)
except Exception:
    tf_keras = None

from tensorflow.keras import layers, Model  

In [ ]:

# ============================================================
# 9) Autoencoder → latent vector
# ============================================================
tf.keras.utils.set_random_seed(42)

input_dim = vec_ldabert_sc.shape[1]
x_in = layers.Input(shape=(input_dim,), name="input")
h1 = layers.Dense(cfg.ae_hidden_dim, activation="relu")(x_in)
z  = layers.Dense(cfg.ae_latent_dim, activation="linear", name="latent")(h1)
h2 = layers.Dense(cfg.ae_hidden_dim, activation="relu")(z)
x_out = layers.Dense(input_dim, activation="linear")(h2)

ae = Model(x_in, x_out, name="autoencoder")
encoder = Model(x_in, z, name="encoder")
ae.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=cfg.ae_patience, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

hist = ae.fit(
    vec_ldabert_sc,
    vec_ldabert_sc,
    validation_split=0.1,
    epochs=cfg.ae_epochs,
    batch_size=cfg.ae_batch_size,
    callbacks=cb,
    verbose=1
)

latent = encoder.predict(vec_ldabert_sc, batch_size=cfg.ae_batch_size, verbose=0).astype(np.float32)
print("latent:", latent.shape)

audit["autoencoder"] = {
    "input_dim": int(input_dim),
    "latent_dim": int(cfg.ae_latent_dim),
    "hidden_dim": int(cfg.ae_hidden_dim),
    "epochs_ran": int(len(hist.history["loss"])),
    "final_loss": float(hist.history["loss"][-1]),
    "final_val_loss": float(hist.history["val_loss"][-1]),
}

# 10) KMeans + silhouette sweep

In [ ]:

# ============================================================
# 10) KMeans + silhouette sweep
# ============================================================
k_range = list(range(cfg.k_min, cfg.k_max + 1))
sil = []
best_k = None
best_s = -1.0

for k in k_range:
    km = KMeans(n_clusters=k, random_state=cfg.k_random_state, n_init="auto")
    labels = km.fit_predict(latent)
    s = float(silhouette_score(latent, labels))
    sil.append({"k": k, "silhouette": s})
    if s > best_s:
        best_s = s
        best_k = k

print("Best K:", best_k, "silhouette:", best_s)

audit["clustering"]["silhouette_sweep"] = sil
audit["clustering"]["best_k"] = best_k
audit["clustering"]["best_silhouette"] = best_s

kmeans = KMeans(n_clusters=best_k, random_state=cfg.k_random_state, n_init="auto")
df["cluster_unsup"] = kmeans.fit_predict(latent).astype(int)
df["cluster_unsup"].value_counts().head()

# 11) Candidate terms per cluster (frequency)

In [ ]:
# ============================================================
# 11) Candidate terms per cluster (frequency) — sumber: unsup_totons
# ============================================================
t0 = time.time()

cluster_terms = {}
for cid, g in df.groupby("cluster_unsup"):
    all_toks = []
    for toks in g["unsup_tokens"].tolist():
        if isinstance(toks, list):
            all_toks.extend(toks)
    cnt = Counter(all_toks)
    # buang toton yang terthen pendek / angka murni
    items = [(w, c) for w, c in cnt.most_common() if len(w) >= 3 and not w.isdigit()]
    cluster_terms[int(cid)] = [w for w, _ in items[:cfg.top_terms_per_cluster]]

audit["cluster_terms"] = {str(k): int(len(v)) for k, v in cluster_terms.items()}
audit["timing_sec"]["cluster_terms"] = round(time.time() - t0, 3)

print("example cluster_terms:", list(cluster_terms.items())[:1])

# 12) Ranking terms per row (cosine) - consistent use of SentenceTransformer

In [ ]:
# ============================================================
# 12) Ranking terms per row (cosine)
# 1: Candidates should NOT be from clusters if they don't appear in the sentence.
# - If intersection(cluster_terms, sentence_tokens) is non-empty -> use intersection (original design)
# - If empty -> fallback to sentence_tokens (instead of "all cluster terms")
# 2: Store row-level audits to make fallback cases easier to trace.
# ============================================================
from functools import lru_cache

t0 = time.time()

# ---------- normalisasi token konsisten ----------
_EDGE_PUNCT = re.compile(r"(^\W+|\W+$)", flags=re.UNICODE)

def _norm_tok(t: str) -> str:
    return _EDGE_PUNCT.sub("", str(t).lower().strip())

def _uniq_preserve(seq):
    seen = set()
    out = []
    for x in seq:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

# Precompute term embeddings per cluster (hemat waktu)
term_emb_by_cluster = {}
for cid, terms in tqdm(cluster_terms.items(), desc="Encode cluster terms"):
    terms_norm = [_norm_tok(t) for t in terms]
    terms_norm = [t for t in terms_norm if t]
    if len(terms_norm) == 0:
        term_emb_by_cluster[int(cid)] = ([], np.zeros((0, sent_emb.shape[1]), dtype=np.float32))
        continue
    emb = st.encode(
        terms_norm,
        batch_size=cfg.st_batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)
    term_emb_by_cluster[int(cid)] = (terms_norm, emb)

# Cache embeddings for token fallback (sentence totons)
_token_emb_cache = {}

def _get_token_embs(tokens):
    tokens = [_norm_tok(t) for t in tokens]
    tokens = [t for t in tokens if t]
    missing = [t for t in tokens if t not in _token_emb_cache]
    if missing:
        emb = st.encode(
            missing,
            batch_size=cfg.st_batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype(np.float32)
        for t, e in zip(missing, emb):
            _token_emb_cache[t] = e
    embs = np.stack([_token_emb_cache[t] for t in tokens], axis=0) if tokens else np.zeros((0, sent_emb.shape[1]), dtype=np.float32)
    return tokens, embs

def rank_terms_for_row(i: int):
    """Return ranked_terms, ranked_sims, fallback_mode, n_intersection, cand_count"""
    row = df.iloc[i]
    cid = int(row["cluster_unsup"])

    s_toks_raw = row["unsup_tokens"] if isinstance(row["unsup_tokens"], list) else []
    s_toks_norm = [_norm_tok(t) for t in s_toks_raw]
    s_toks_norm = [t for t in s_toks_norm if t]
    s_toks_norm = _uniq_preserve(s_toks_norm)
    s_set = set(s_toks_norm)

    terms_cluster, emb_cluster = term_emb_by_cluster.get(cid, ([], np.zeros((0, sent_emb.shape[1]), dtype=np.float32)))

    # --- kandidat from cluster HARUS muncul di sentence ---
    inter = [t for t in terms_cluster if t in s_set]
    if len(inter) > 0:
        # use embedding cluster term, tapi hanya for term yang ada di kalimat
        idx = [terms_cluster.index(t) for t in inter]
        cand_terms = inter
        cand_emb = emb_cluster[idx, :]
        fallback_mode = "cluster_intersection"
        n_intersection = len(inter)
    else:
        # FIX UTAMA: fallback must kandidat from kalimat
        cand_terms, cand_emb = _get_token_embs(s_toks_norm)
        fallback_mode = "sentence_tokens"
        n_intersection = 0

    if cand_emb.shape[0] == 0:
        return [], [], fallback_mode, n_intersection, 0

    # cosine similarity karena embedding sudah dinormalisasi => dot product
    sims = cand_emb @ sent_emb[i].reshape(-1, 1)
    sims = sims.reshape(-1)

    k = min(cfg.topn_rank, len(cand_terms))
    top_idx = np.argpartition(-sims, kth=k-1)[:k]
    top_idx = top_idx[np.argsort(-sims[top_idx])]

    ranked_terms = [cand_terms[j] for j in top_idx]
    ranked_sims  = [float(sims[j]) for j in top_idx]
    return ranked_terms, ranked_sims, fallback_mode, n_intersection, len(cand_terms)

ranked_terms_list, ranked_sims_list = [], []
fallback_mode_list, n_intersection_list, cand_count_list = [], [], []

rows_intersection = 0
rows_fallback_sentence = 0

for i in tqdm(range(len(df)), desc="Ranking per row"):
    t, s, mode, n_inter, n_cand = rank_terms_for_row(i)
    ranked_terms_list.append(t)
    ranked_sims_list.append(s)
    fallback_mode_list.append(mode)
    n_intersection_list.append(n_inter)
    cand_count_list.append(n_cand)
    if mode == "cluster_intersection":
        rows_intersection += 1
    else:
        rows_fallback_sentence += 1

df["terms_ranked"] = ranked_terms_list
df["cos_ranked"]   = ranked_sims_list
df["top_aspect_terms_unsup"] = df["terms_ranked"].apply(lambda x: x[:cfg.topn_keep])
df["top_aspect_cos_unsup"]   = df["cos_ranked"].apply(lambda x: x[:cfg.topn_keep])

# audit columns (row-level)
df["fallback_mode"] = fallback_mode_list
df["n_intersection"] = n_intersection_list
df["cand_count"] = cand_count_list

t1 = time.time()
audit["ranking"]["rows_intersection"] = int(rows_intersection)
audit["ranking"]["rows_fallback_sentence_tokens"] = int(rows_fallback_sentence)
audit["ranking"]["token_emb_cache_size"] = int(len(_token_emb_cache))
audit["timing_sec"]["ranking"] = round(t1-t0, 3)

df[["unsup_filtered","cluster_unsup","fallback_mode","n_intersection","top_aspect_terms_unsup"]].head(8)

# 13) POS filter (NOUN/PROPN) via Stanza

In [ ]:
from functools import lru_cache

t0 = time.time()

def build_stanza_pipeline(lang: str):
    # POS from konteks kalimat: totonize + pos cukup
    return stanza.Pipeline(lang=lang, processors="tokenize,pos", tokenize_no_ssplit=True, verbose=False)

try:
    nlp_id = build_stanza_pipeline(cfg.stanza_lang_primary)
except Exception as e:
    print("Gagal init stanza ID:", e)
    nlp_id = None

try:
    nlp_en = build_stanza_pipeline(cfg.stanza_lang_fallback)
except Exception as e:
    print("Gagal init stanza EN:", e)
    nlp_en = None

if (nlp_id is None) and (nlp_en is None):
    raise RuntimeError(
        "Stanza pipeline gagal (ID dan EN). "
        "Jalankan: `python -m stanza.download id` (dan opsional `python -m stanza.download en`), "
        "lalu restart kernel."
    )

_EDGE_PUNCT = re.compile(r"(^\W+|\W+$)", flags=re.UNICODE)

def _norm_tok(t: str) -> str:
    return _EDGE_PUNCT.sub("", str(t).lower().strip())

@lru_cache(maxsize=20000)
def stanza_pos_map_for_sentence(sentence: str):
    sent = str(sentence).strip()
    if not sent:
        return {}

    # coba ID dulu
    if nlp_id is not None:
        doc = nlp_id(sent)
        m = {}
        for s in doc.sentences:
            for w in s.words:
                key = _norm_tok(w.text)
                if key and key not in m:
                    m[key] = w.upos
        if m:
            return m

    # fallback EN
    if nlp_en is not None:
        doc = nlp_en(sent)
        m = {}
        for s in doc.sentences:
            for w in s.words:
                key = _norm_tok(w.text)
                if key and key not in m:
                    m[key] = w.upos
        return m

    return {}

KEEP_UPOS = {"NOUN","PROPN"}

def filter_terms_pos_context(row):
    pos_map = stanza_pos_map_for_sentence(row["unsup_filtered"])
    kept = []
    removed = []
    removed_upos = []
    for t in (row["top_aspect_terms_unsup"] if isinstance(row["top_aspect_terms_unsup"], list) else []):
        key = _norm_tok(t)
        up = pos_map.get(key)
        if up in KEEP_UPOS:
            kept.append(key)
        else:
            removed.append(key)
            removed_upos.append(up if up is not None else "NA")
    return kept, removed, removed_upos

tmp = df.progress_apply(filter_terms_pos_context, axis=1)
df["aspect_terms_unsup"] = tmp.apply(lambda x: x[0])
df["pos_removed_terms"] = tmp.apply(lambda x: x[1])
df["pos_removed_upos"]  = tmp.apply(lambda x: x[2])

# audit POS
pos_kept = df["aspect_terms_unsup"].apply(len)
audit.setdefault("pos_filter", {})
audit["pos_filter"]["avg_kept"] = float(pos_kept.mean())
audit["pos_filter"]["median_kept"] = float(pos_kept.median())
audit["pos_filter"]["rows_empty_after_pos"] = int((pos_kept==0).sum())

t1 = time.time()
audit["timing_sec"]["pos_filter"] = round(t1-t0, 3)

df[["unsup_filtered","top_aspect_terms_unsup","aspect_terms_unsup","pos_removed_terms"]].head(8)

# 14) Save outputs + audit

In [ ]:
# ============================================================
# 14) Save outputs + audit (CSV + JSON + audit-rowlevel)
# ============================================================
# 14.1 full output
cols_out1 = list(join_cols) + [
    "unsup_filtered","unsup_text_norm","cluster_unsup",
    "terms_ranked","cos_ranked",
    "top_aspect_terms_unsup","top_aspect_cos_unsup",
    "aspect_terms_unsup",
    "fallback_mode","n_intersection","cand_count",
    "pos_removed_terms","pos_removed_upos",
]
cols_out1 = [c for c in cols_out1 if c in df.columns]
out1 = df[cols_out1].copy()
out1.to_csv(OUT_DIR / "unsup_terms_ranked_full.csv", index=False, encoding="utf-8")

# 14.2 compact
if len(join_cols) > 0:
    out2 = df[join_cols + ["cluster_unsup","aspect_terms_unsup"]].copy()
else:
    out2 = df[["cluster_unsup","aspect_terms_unsup"]].copy()
out2.to_csv(OUT_DIR / "unsup_aspect_terms_compact.csv", index=False, encoding="utf-8")

# 14.3 audit rowlevel (ringkas for tracing kasus aneh)
audit_rows = pd.DataFrame({
    "row_id": np.arange(len(df)),
    "cluster_unsup": df["cluster_unsup"].astype(int),
    "fallback_mode": df.get("fallback_mode", "NA"),
    "n_intersection": df.get("n_intersection", np.nan),
    "cand_count": df.get("cand_count", np.nan),
    "n_tokens": df["unsup_tokens"].apply(lambda x: len(x) if isinstance(x, list) else 0),
    "n_ranked": df["terms_ranked"].apply(lambda x: len(x) if isinstance(x, list) else 0),
    "n_after_pos": df["aspect_terms_unsup"].apply(lambda x: len(x) if isinstance(x, list) else 0),
})
for k in join_cols:
    if k in df.columns:
        audit_rows[k] = df[k]
audit_rows.to_csv(OUT_DIR / "audit_rowlevel.csv", index=False, encoding="utf-8")

# 14.4 audit json
save_json(audit, OUT_DIR / "audit_unsup_refactor_v6.json")

print("Saved:")
print(" -", (OUT_DIR / "unsup_terms_ranked_full.csv").resolve())
print(" -", (OUT_DIR / "unsup_aspect_terms_compact.csv").resolve())
print(" -", (OUT_DIR / "audit_rowlevel.csv").resolve())
print(" -", (OUT_DIR / "audit_unsup_refactor_v6.json").resolve())